# SNNAI v6.6.2 fair_compare (all features enabled)

Measures current SNN val_ppl with `use_sequence_recurrent=True`, `use_positional_encoding=True`, `use_hippocampus_gate=True`, `use_spiking_attention=True`, vs a Transformer baseline.
Conditions match v6.5.7: BPE vocab 2048, WikiText-2 + Tiny Shakespeare, seq_len 128, batch 32, time_steps 20, epochs 5, T4 GPU.


In [ ]:
# Install and clone
!pip install -q torch numpy snntorch
!rm -rf snnai
!git clone --depth 1 --branch v6.6.2 https://github.com/sabunosuke1008-create/snnai.git
import sys
sys.path.insert(0, 'snnai')


In [ ]:
import os, json, time
import torch
import requests
from snnai.utils.download_corpus import download_wikitext2
from snnai.modules.language.bpe_tokenizer import BPETokenizer
from snnai.modules.language.large_scale_lm import LargeScaleSNNLM
from snnai.benchmarks.transformer_comparison import TransformerBaseline, fair_compare

t0 = time.time()

# 1) corpus: Tiny Shakespeare + WikiText-2 (raw)
corpus = requests.get('https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt').text
wt_root = download_wikitext2(dest_dir='/kaggle/working/wikitext-2', timeout=120)
if wt_root is not None:
    wt_text = ''.join(
        open(os.path.join(wt_root, f)).read()
        for f in ['wiki.train.raw', 'wiki.valid.raw', 'wiki.test.raw']
        if os.path.exists(os.path.join(wt_root, f))
    )
    if wt_text:
        corpus = corpus + '\n' + wt_text
print('corpus length', len(corpus))

# 2) BPE vocab 2048
bpe = BPETokenizer([corpus], vocab_size=2048, max_train_bytes=2_000_000)
print('bpe vocab', bpe.vocab_size)

# 3) models (all SNN features ON)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device', device)
snn = LargeScaleSNNLM(
    vocab_size=bpe.vocab_size, embed_dim=128, hidden_dim=512, num_layers=3,
    use_sequence_recurrent=True, use_positional_encoding=True,
    use_hippocampus_gate=True, use_spiking_attention=True,
    hippocampus_capacity=64, max_seq_len=128, output_mode='mem_last',
)
tf = TransformerBaseline(bpe.vocab_size, d_model=512, nhead=4, num_layers=3, dim_feedforward=1024)
snn_p = sum(p.numel() for p in snn.parameters())
tf_p = sum(p.numel() for p in tf.parameters())
print('snn params', snn_p, 'tf params', tf_p)

# 4) fair compare (epochs=5 to match v6.5.7)
res = fair_compare(corpus, bpe, snn, tf, epochs=5, seq_len=128, batch_size=32,
                  time_steps=20, device=device, save_dir='/kaggle/working')
print('SNN train_ppl', res['snn_history']['train_ppl'])
print('SNN val_ppl', res['snn_history']['val_ppl'])
print('TF  train_ppl', res['transformer_history']['train_ppl'])
print('TF  val_ppl', res['transformer_history']['val_ppl'])
print('snn_latency', res['snn_latency'], 'tf_latency', res['transformer_latency'])
print('elapsed_sec', round(time.time() - t0, 1))

summary = {
    'config': 'all_features_on',
    'device': device,
    'snn_params': snn_p,
    'tf_params': tf_p,
    'snn_train_ppl': res['snn_history']['train_ppl'],
    'snn_val_ppl': res['snn_history']['val_ppl'],
    'tf_train_ppl': res['transformer_history']['train_ppl'],
    'tf_val_ppl': res['transformer_history']['val_ppl'],
    'snn_latency': res['snn_latency'],
    'tf_latency': res['transformer_latency'],
    'snn_bleu1': res['snn_generation'].get('bleu_1_mean'),
    'tf_bleu1': res['transformer_generation'].get('bleu_1_mean'),
    'elapsed_sec': round(time.time() - t0, 1),
}
print(json.dumps(summary, indent=2))
with open('/kaggle/working/fair_compare_v662.json', 'w') as f:
    json.dump(summary, f, indent=2)
